In [1]:
import copy
import json
import numpy as np

import glob
import os
from pymatgen.electronic_structure.plotter import DosPlotter
from pymatgen.electronic_structure import dos
from pymatgen.electronic_structure.core import Spin
from ase.io import read

import matplotlib.pyplot as plt
import matplotlib
from scipy.interpolate import make_interp_spline
import re

splitbycap = '[A-Z][^A-Z]*'

matplotlib.rc('axes',edgecolor='k')
matplotlib.rc('xtick',labelsize=10)
matplotlib.rc('ytick',labelsize=10)
matplotlib.rc('axes',labelsize=16)
matplotlib.rc('axes',titlesize=16)


In [2]:
def get_data(filename):
    with open(filename,'r') as f:
        data = json.load(f)
    return data

def write_data(filename, data):
    with open(filename,'w') as f:
        json.dump(data, f)

def getMAD(list1,list2):
    if len(list1)!=len(list2):
        print('lists need to have the same dimensions')
        return 
    MAD_list = []
    for i in range(len(list1)):
        x1 = list1[i]
        x2 = list2[i]
        if not np.isnan(x1) and not np.isnan(x2):
            MAD_list.append(np.abs(x1-x2))
    MAD = np.sum(MAD_list)/len(MAD_list)
    return MAD, len(MAD_list)

def getMD(list1,list2):
    if len(list1)!=len(list2):
        print('lists need to have the same dimensions')
        return 
    MD_list = []
    for i in range(len(list1)):
        x1 = list1[i]
        x2 = list2[i]
        if not np.isnan(x1) and not np.isnan(x2):
            MD_list.append(x1-x2)
    MD = np.sum(MD_list)/len(MD_list)
    return MD, len(MD_list)


In [3]:
raw_phase_diagram_energy_dict = get_data('data/DFT results/alloy_space_overall_dict_EMPTY.json')

DFTatMLIP_result_1 = get_data('data/DFT results/DFTatMLIP_result_1.json')
DFTatMLIP_result_2 = get_data('data/DFT results/DFTatMLIP_result_2.json')
DFTatMLIP_result_3 = get_data('data/DFT results/DFTatMLIP_result_3.json')
DFTatMLIP_result_4 = get_data('data/DFT results/DFTatMLIP_result_4.json')
DFTatMLIP_result_5 = get_data('data/DFT results/DFTatMLIP_result_5.json')

All_DFTatMLIP_raw_results = {}
All_DFTatMLIP_raw_results.update(DFTatMLIP_result_1)
All_DFTatMLIP_raw_results.update(DFTatMLIP_result_2)
All_DFTatMLIP_raw_results.update(DFTatMLIP_result_3)
All_DFTatMLIP_raw_results.update(DFTatMLIP_result_4)
All_DFTatMLIP_raw_results.update(DFTatMLIP_result_5)

In [4]:
figure_size = [6,6]

frac_str_tuple = ('0', '25', '50', '75', '100')
frac_float = [0,0.25,0.5,0.75,1]

all_alloy_sets = []

save_folder = 'manuscript_figures'
if not os.path.exists(save_folder):
    os.mkdir(save_folder)


alloy_halide_perovskites = ['KRbMgNiF6', 'Cs2SrPbBr6']#, 'Rb2MnNiF6']
figure_numbers = ['4a', '4b']#, '4c']

count = 0
for alloy_series, figure_number in zip(alloy_halide_perovskites, figure_numbers):
    v = raw_phase_diagram_energy_dict[alloy_series]
    all_alloy_sets.append(alloy_series)
    els = re.findall(splitbycap, alloy_series)
    if len(els) == 5:
        A = els[0]+els[1]
        B1 = els[2]
        B2 = els[3]
        X = els[4].split('6')[0]
        name = A+'('+B1+'$_{1-x}$'+B2+'$_{x}$'+')'+'$_2$'+X+'$_{6}$'
        filename = 'Figure_' + figure_number + '_' + A+'('+B1+'_(1-x)'+B2+'_(x)'+')_2'+X+'_6'
    elif len(els) == 4:
        A = els[0].split('2')[0]
        B1 = els[1]
        B2 = els[2]
        X = els[3].split('6')[0]
        name = A+B1+'$_{1-x}$'+B2+'$_{x}$'+X+'$_{3}$'
        filename = 'Figure_' + figure_number + '_' + A+B1+'_(1-x)'+B2+'_(x)'+X+'_3'

    final_path = os.path.join(save_folder, filename)
    if not os.path.exists(final_path):
        
        Y = []
        for frac in frac_str_tuple:
            formula = v[frac]['formula']
            Eg = All_DFTatMLIP_raw_results[formula]['dos']['band_gap']['energy']
            Y.append(Eg)
        
        plt.figure(figsize=figure_size)
    
        plt.plot([0,1],[0,0], 'k--', zorder=1)
    
        if np.nan not in Y:
            # X_Y_Spline = make_interp_spline(frac_float, Y) #make a smooth curve out of discreet points at fractions X and E_mix Y
            # x = np.linspace(frac_float[0], frac_float[-1], 500) #generate a semi continuous series of values between 0 and 1 (first and last entry of X)
            # y = X_Y_Spline(x)
            # plt.plot(x,y, 'k', alpha=1, zorder=2)
            plt.plot(frac_float, Y, 'k', alpha=1, zorder=2)
        else:
                            
            new_x ,new_y = [], []
            for i in range(len(frac_float)):
                if not np.isnan(Y[i]):
                    new_x.append(frac_float[i])
                    new_y.append(Y[i])
            plt.plot(new_x, new_y, 'k', marker='x', alpha=1, zorder=2)
    
        plt.scatter(frac_float, Y, marker='x', c='r', alpha=1, zorder=2)
        count += 1
    
        plt.xlabel('x, mol fraction')
        plt.ylabel('PBE $E_g$ ($eV$)')
        plt.xticks([0,0.25,0.5,0.75,1])
        plt.xlim([0,1])
        # plt.ylim([-1.2,0.2])
        # plt.ylim([-0.06,0.05])
        plt.grid(False)
        
        plt.title(name)
        plt.savefig(final_path)
        plt.close()


## Plot elemental DOS (Combined by orbitals)

In [5]:
def split_by_elements(formula):
    '''
    :param: formula (string)
    :return: list of tuples containing elemental identity and stoichiometry
    '''
    splitbycap = '[A-Z][^A-Z]*'
    split_els = re.findall(splitbycap, formula) #split formula by capital letter
    els = []
    for split_el in split_els:
        #separate string into element and stoichiometry number
        el = [''.join([i for i in split_el if not i.isdigit()]), ''.join([i for i in split_el if i.isdigit()])] 
        if el[1]=='':
            el[1]='1'
        els.append(tuple(el))
    return els

def make_el_spin_plot_list(els, orbs = ['all']):
    '''
    Takes output from split_by_element and generates a list of (element, orbs, spin)
    :param els: 
    :return: list of tuple (element, orbs, spin)
    '''
    el_tuple_list = []
    for el_tuple in els:
        for orb in orbs:
            el_tuple_list.append((el_tuple[0], orb, 'up'))
            el_tuple_list.append((el_tuple[0], orb, 'down'))
    return el_tuple_list

def generate_plot_initialization_list(formula, path, combine_orbs=False, B_only=False):
    
    d_block = ['Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
               'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd',
               'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg',
               'Rf', 'Db', 'Sg', 'Bh', 'Hs', 'Mt', 'Ds', 'Rg', 'Cn']
    d_orbs = ['dxy', 'dxz', 'dyz', 'dz2', 'dx2']
    d_orbs_combined = ['d_total']
    halides = ['F', 'Cl', 'Br', 'I']
    halide_orbs = ['s', 'px', 'py', 'pz']
    halide_orbs_combined = ['s_total', 'p_total']
    
    dos_path_list = []
    plot_els_spin_list = []
    plot_types = []
    formula_list = []
    formula_list.append(formula)
    
    if B_only:
        els_tuple_full = split_by_elements(formula)
        if len(els_tuple_full)==3:
            els_tuple = [els_tuple_full[1]]
        elif len(els_tuple_full)==4:
            els_tuple = [els_tuple_full[2]] if els_tuple_full[0][1]=='1' else [els_tuple_full[1], els_tuple_full[2]]
        elif len(els_tuple_full)==5:
            els_tuple = [els_tuple_full[2], els_tuple_full[3]]
    else:
        els_tuple = split_by_elements(formula)
    els_list = [tup[0] for tup in els_tuple]
    formula_list = formula_list+els_list
    
    dos_path_list.append(path)
    
    els_spin_plot_list = make_el_spin_plot_list(els_tuple, orbs = ['all'])
    plot_els_spin_list.append(els_spin_plot_list)
    plot_types.append('all')
    
    for i in range(len(els_tuple)):
        dos_path_list.append(path)
        el_tuple = els_tuple[i]
        el = el_tuple[0]
        if el in halides:
            orb_list = halide_orbs if combine_orbs==False else halide_orbs_combined
        elif el in d_block:
            orb_list = ['s']+d_orbs if combine_orbs==False else ['s_total']+d_orbs_combined
        else:
            orb_list = halide_orbs+d_orbs if combine_orbs==False else halide_orbs_combined+d_orbs_combined
        els_spin_plot_list = make_el_spin_plot_list([el_tuple], orbs = orb_list)
        plot_els_spin_list.append(els_spin_plot_list)
        plot_types.append('orb')
    
    return formula_list, dos_path_list, plot_els_spin_list, plot_types

In [6]:
def orbital_resolved_dos_plot_initialization(formula, path, combine_orbs=False, B_only=False):
    
    d_block = ['Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
               'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd',
               'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg',
               'Rf', 'Db', 'Sg', 'Bh', 'Hs', 'Mt', 'Ds', 'Rg', 'Cn']
    d_orbs = ['dxy', 'dxz', 'dyz', 'dz2', 'dx2']
    d_orbs_combined = ['d_total']
    halides = ['F', 'Cl', 'Br', 'I']
    halide_orbs = ['s', 'px', 'py', 'pz']
    halide_orbs_combined = ['s_total', 'p_total']
    
    dos_path = path
    plot_els_spin = []
    plot_type = 'orb'
    formula = formula
    
    if B_only:
        els_tuple_full = split_by_elements(formula)
        if len(els_tuple_full)==3:
            els_tuple = [els_tuple_full[1]]
        elif len(els_tuple_full)==4:
            els_tuple = [els_tuple_full[2]] if els_tuple_full[0][1]=='1' else [els_tuple_full[1], els_tuple_full[2]]
        elif len(els_tuple_full)==5:
            els_tuple = [els_tuple_full[2], els_tuple_full[3]]
    else:
        els_tuple = split_by_elements(formula)
    

    for i in range(len(els_tuple)):
        el_tuple = els_tuple[i]
        el = el_tuple[0]
        if el in halides:
            orb_list = halide_orbs if combine_orbs==False else halide_orbs_combined
        elif el in d_block:
            orb_list = ['s']+d_orbs if combine_orbs==False else ['s_total']+d_orbs_combined
        else:
            orb_list = halide_orbs+d_orbs if combine_orbs==False else halide_orbs_combined+d_orbs_combined
        els_spin_plot_list = make_el_spin_plot_list([el_tuple], orbs = orb_list)
        plot_els_spin = plot_els_spin+ els_spin_plot_list
    
    return dos_path, plot_els_spin, plot_type

In [7]:
color_dict = {'Pd':'silver', 'Rb':'purple', 'Zn':'tab:gray', 'Fe':'tab:brown', 'Ca':'cadetblue', 'I':'indigo',
              'Cl':'lime', 'Cd':'magenta', 'Ba':'springgreen', 'Cs':'turquoise', 'F':'lightsteelblue', 'K':'blueviolet',
              'Co':'navy', 'Mn':'darkmagenta', 'Sr':'lawngreen', 'Pb':'slategrey', 'Ni':'grey', 'Br':'peru','Ti':'cyan',
              'Na':'khaki', 'V':'red', 'Cu':'blue', 'Cr':'darkslateblue', 'Mg':'orange', 'Sn':'lightgrey', 'Ag':'silver',
              'Pt':'cyan', 'Li':'khaki', 'Sc':'darkgrey', 'Au':'gold'}

AEM_color_orbs = {'s_total': '#1f77b4', 'p_total': '#17becf', 'd_total':'#2ca02c'}

PTM_color_orbs = {'s_total': '#d62728', 'p_total': '#ff7f0e', 'd_total':'#9467bd'}

TM_color_orbs = {'s_total': '#8c564b', 'p_total': '#7f7f7f', 'd_total':'#e377c2'}

B1_color_orbs = {'s_total': '#1f77b4', 'p_total': '#17becf', 'd_total':'#2ca02c'}

B2_color_orbs = {'s_total': '#d62728', 'p_total': '#ff7f0e', 'd_total':'#9467bd'}

In [8]:
dos_paths = glob.glob('data/DFT dos/*_complete_dos.json')

dos_path_dict = {}
for path in dos_paths:
    comp = path.split(os.sep)[-1].split('_')[0]
    dos_path_dict[comp]=path

In [9]:
# dos_helper.py is a helper file written by Nicholas Singstock, Ph.D. (thank you!)
# contains functions for plotting DOS and PDOS data. It is imported here to use the functions defined in it.
from dos_helper import set_rc_params, get_plot, fill_zeros 

set_rc_params()

{'axes.linewidth': 1.5,
 'axes.unicode_minus': False,
 'figure.dpi': 100,
 'font.size': 16,
 'font.family': 'sans-serif',
 'font.sans-serif': 'Verdana',
 'legend.frameon': False,
 'legend.handletextpad': 0.2,
 'legend.handlelength': 0.6,
 'legend.fontsize': 12,
 'legend.columnspacing': 0.8,
 'mathtext.default': 'regular',
 'savefig.bbox': 'tight',
 'xtick.labelsize': 16,
 'ytick.labelsize': 16,
 'xtick.major.size': 6,
 'ytick.major.size': 6,
 'xtick.major.width': 1.5,
 'ytick.major.width': 1.5,
 'xtick.top': False,
 'xtick.bottom': True,
 'ytick.right': True,
 'ytick.left': True,
 'xtick.direction': 'out',
 'ytick.direction': 'out',
 'axes.edgecolor': 'black'}

In [11]:
foldername = 'manuscript_figures'
frac_str_list = ['0', '25', '50', '75', '100']
missing_comp = []

#whether to save
elim = [-3, 8]
dens_range = [-1/3000*2,1/3000*2]
save = True

smear = 0.05
zero_fermi = True
zero_width = 0.02

dos_lines = []#(-6.55,'k'),]#(-3.8,'tab:green')]

alloys_list = ['KRbMgNiF6', 'Cs2SrPbBr6']#, 'Rb2MnNiF6'] #add more alloy series here
figure_numbers = ['5', '6']#, 'S'] #add correspnding figure numbers here len(figure_numbers) == len(alloys_list)

for alloy, figure_number in zip(alloys_list, figure_numbers):
    dos_path_list = []
    plot_els_spin_list = []
    formula_list = []
    plot_types = []
    for frac_str in frac_str_list:
        v = raw_phase_diagram_energy_dict[alloy][frac_str]
        formula = v['formula']
        formula_list.append(formula)
        if not formula in dos_path_dict:
            missing_comp.append(formula)
            dos_path_list.append(None)
            plot_els_spin_list.append(None)
            plot_types.append('orb')
        else:
            dos_path, plot_els_spin, plot_type = orbital_resolved_dos_plot_initialization(formula, dos_path_dict[formula], combine_orbs=True, B_only=True)
            dos_path_list.append(dos_path)
            plot_els_spin_list.append(plot_els_spin)
            plot_types.append(plot_type)
    B1 = ''
    B2 = ''
    for tup in plot_els_spin_list[1]:
        if tup!=None:
            if len(B1)==0 and len(B2)==0:
                B1 = tup[0]
            elif len(B1)>0 and len(B2)==0 and tup[0]!=B1:
                B2 = tup[0]
    # print(B1, B2)
    
    nplots = len(dos_path_list)
    
    save_name = os.path.join(foldername, 'Figure_' + figure_number + '_' + alloy + '.png')
    
    skip_atom_number = False
    all_jdos = {}
    fig, axes = plt.subplots(1, nplots, figsize = (3*nplots, 5), sharey = 'row', dpi = 600 if save else 100)

    M_orbital = 'dz2'
    
    if not os.path.exists(save_name) or True:
        for i, file in enumerate(dos_path_list):
        #     file = './' + file
    
            # alpha = [0.25, 0.25,
            #          0.45, 0.45,
            #          0.5, 0.5,
            #          ]
    
            
            plot_type = plot_types[i]
            
            dos_to_plot = plot_els_spin_list[i]
            if plot_type == 'all':
                colors = [color_dict[j[0]] for j in plot_els_spin_list[i]] if plot_els_spin_list[i]!=None else None
            elif plot_type == 'orb':
                colors = []
                for j in plot_els_spin_list[i]:
                    if j!=None:
                        if j[0]==B1:
                            colors = colors + [B1_color_orbs[j[1]]]
                        elif j[0]==B2:
                            colors = colors + [B2_color_orbs[j[1]]]
                        else:
                            print('something\'s wrong')
                        # if j[0] in d_block:
                        #     colors = colors + [TM_color_orbs[j[1]]]
                        # elif j[0] in AEM_list:
                        #     colors = colors + [AEM_color_orbs[j[1]]]
                        # elif j[0] in PTM_list:
                        #     colors = colors + [PTM_color_orbs[j[1]]]
                    else:
                        colors = None
            alpha = [0.5 for j in plot_els_spin_list[i]] if plot_els_spin_list[i]!=None else None
            # colors = ['tab:red', 'tab:red',
            #           'tab:orange','tab:orange',                  
            #           'tab:blue', 'tab:blue',                  
            #           ]
        
            # centers = [[0,1],[2,3,],[4,5],]
        
            axid = i
    
            ax = axes[axid]
    
        #    assert False
    
            # setup plotted and add dos objects 
            plotter = DosPlotter(sigma = smear, zero_at_efermi=zero_fermi)  # energies are already zeroed!
            names = []
            if file != None:
                with open(file,'r') as f:
                    jdos = json.load(f)
                all_jdos[file] = jdos
                
                efermi = jdos['Efermi']
                
                for pdos in dos_to_plot:
                    spin = pdos[-1]
            #        x = jdos['up']['Total']
            #        xd = jdos['down']['Total']
            #        y = jdos['up']['Energy']
                    
                    normalization_constant = np.sum(jdos[spin]['Total'])
                    
                    if pdos[0] == 'Total':
                        dens = jdos[spin]['Total']
                        energy, density = fill_zeros(jdos[spin]['Energy'], dens, zero_width)
            #            if not center_fermi: 
                        energy = [e + efermi for e in energy]
                        name = 'Total '+spin
                    else:
                        dens = jdos[spin][pdos[0]][pdos[1]] # Atom_# + orbital
                        energy, density = fill_zeros(jdos[spin]['Energy'], dens, zero_width)
            #            if not center_fermi:
                        energy = [e + efermi for e in energy] # Always convert back to absolute Fermi scale and let pmg re-zero
                        if skip_atom_number:
                            orbital = pdos[1]
                            if len(orbital) > 1:
                                orbital = orbital[0] + '$_{'+orbital[1:]+'}$'
                            name = pdos[0]+' '+ pdos[1].replace('_total', '') +' '+spin
                        else:
                            name = pdos[0]+' '+ pdos[1].replace('_total', '') +' '+spin #MODIFIED only need this line because it's a bulk calculation
            #                 orbital = pdos[0].split('_')[1]
            # #                 print('PDOS',pdos[0])
            #                 if len(orbital) > 1:
            #                     orbital = orbital[0] + '$_{'+orbital[1:]+'}$'
            #                 name = pdos[0].split('_')[0]+'('+orbital+') '+pdos[1] +' '+spin
            #                 print(orbital)
        
                    pmg_jdos = dos.Dos(efermi, #if center_fermi else 0.0, 
                                       energy,
                                       {Spin.up if spin == 'up' else Spin.down: [d/normalization_constant for d in density]})#density})
                    plotter.add_dos(name, pmg_jdos)
                    names.append(name)
            # else:
            #     plotter.add_dos()
    
                # reorder colors and alpha lists to match dos keys
                old_names = names
            #     names, colors = zip(*sorted(zip(names, colors)))
                colors = list(colors)
                names = old_names
            #     names, alpha = zip(*sorted(zip(names, alpha)))
                alpha = list(alpha)
        
                # print('\n\n', names, colors, alpha)
        
                plot, debug = get_plot(plotter, energy_lim=elim, colors=colors, alpha = alpha,
                                density_lim = dens_range,# if axid in [0,1,2,3,4] else None,
                                normalize_density=False, ax = ax, 
                                lloc = (0.92, 1.05), #if axid == 3 else (1.0, 1.0), 
                                lcol = 1, lframe = True,
                                ylabel = True if axid == 0 else False, density_ticks = False, 
            #                    mark_fermi = efermi if axid in [0,1,2,3] else None,
                                mark_fermi = True, fill_to_efermi = True, pdos_label = True,
                                dos_lines = dos_lines, show_legend = True)#,
                                # show_band_centers = centers)
            else:
                ax.plot([],[])
    
    
        # colors = ['tab:red', 'tab:orange', 'tab:blue']
        # atoms = ['Mo', 'S', 'Cu']
        for k in range(nplots):
            axes[k].set_title(formula_list[k])
    
    
    
        if save:
            fig.savefig(save_name)
            plt.close()
    else:
        plt.close()

            
    